# MolDA Dataset Generation — Summary & Statistics

이 노트북은 MolDA 데이터셋 파이프라인의 전체 통계를 요약합니다.

**파이프라인 구조 (2026-04 기준):**
- `Raw/{tag}/step1/` — Step 1 (Download & Process) per-task Arrow
- `Raw/{tag}/step2/` — Step 2 (Decontamination) per-task Arrow (dedup on일 때만)
- `Processed/{tag}/{Train,Val,Test}/` — Step 3 (Concatenate & Map) 최종 데이터셋
  - 하나의 디렉터리에 `*_smiles` / `*_selfies` 듀얼 컬럼으로 저장됨

**내용:**
1. Source Dataset별 원본 샘플 수
2. Task별 처리 후 샘플 수 (train/val/test, step1 & step2 비교)
3. Preprocessing 과정의 탈락(rejection) 수 및 비율
4. Cross-Source Decontamination 제거 통계
5. Step 3 SELFIES 변환 실패 filter
6. 최종 데이터셋 요약

**참고 논문:**
- **Mol-LLM** (arXiv:2502.02810): "tasks present in both datasets are deduplicated to ensure that molecules included in the test set of one dataset do not appear in the training set of the combined dataset."
- **SMolInstruct** (osunlp/SMolInstruct): 14 task types, ~3.4M samples
- **Mol-Instructions** (zjunlp/Mol-Instructions): reaction prediction, retrosynthesis, property prediction
- **ChEBI-20** (liupf/ChEBI-20-MM): molecule captioning / generation


In [ ]:
import os
import warnings
from collections import defaultdict

import pandas as pd
import pyarrow.ipc as ipc

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.1f}")

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
PROJECT_ROOT = "/opt/EMNLP_MolDA/New_MolDA"
RAW_DATA_ROOT = os.path.join(PROJECT_ROOT, "dataset/Raw")
PROCESSED_DATA_ROOT = os.path.join(PROJECT_ROOT, "dataset/Processed")

# 분석할 data_tag (configs/download/*.yaml의 data_tag + resolve_data_tag 적용값)
DATA_TAG = "raw_v1_10x_rephrase"

TAG_RAW_ROOT = os.path.join(RAW_DATA_ROOT, DATA_TAG)          # Raw/{tag}/
STEP1_ROOT = os.path.join(TAG_RAW_ROOT, "step1")              # per-task (Step 1)
STEP2_ROOT = os.path.join(TAG_RAW_ROOT, "step2")              # per-task (Step 2)
PROCESSED_ROOT = os.path.join(PROCESSED_DATA_ROOT, DATA_TAG)  # final concat

print(f"Project Root:    {PROJECT_ROOT}")
print(f"Data tag:        {DATA_TAG}")
print(f"Step 1 per-task: {STEP1_ROOT}  (exists={os.path.isdir(STEP1_ROOT)})")
print(f"Step 2 per-task: {STEP2_ROOT}  (exists={os.path.isdir(STEP2_ROOT)})")
print(f"Processed final: {PROCESSED_ROOT}  (exists={os.path.isdir(PROCESSED_ROOT)})")

# Step 2 완료 마커 확인
STEP2_MARKER = os.path.join(TAG_RAW_ROOT, ".step2_done")
print(f"Step 2 marker:   {STEP2_MARKER}  (exists={os.path.exists(STEP2_MARKER)})")


In [ ]:
# ---------------------------------------------------------------------------
# Helper: Arrow 파일에서 row count 읽기
# ---------------------------------------------------------------------------
def count_arrow_rows(dataset_dir):
    """Arrow 파일들의 총 row 수를 반환. 디렉터리가 없으면 None."""
    if not os.path.isdir(dataset_dir):
        return None
    arrow_files = sorted(
        f for f in os.listdir(dataset_dir)
        if f.startswith("data-") and f.endswith(".arrow")
    )
    if not arrow_files:
        return 0
    total = 0
    for af in arrow_files:
        path = os.path.join(dataset_dir, af)
        try:
            reader = ipc.open_stream(path)
            for batch in reader:
                total += batch.num_rows
        except Exception:
            try:
                table = ipc.open_file(path).read_all()
                total += len(table)
            except Exception:
                pass
    return total


def scan_per_task_counts(per_task_root):
    """per_task_root 아래의 모든 {task}_subtask-{idx}_{split} 디렉터리를 스캔하여
    task별 train/val/test 카운트를 dict로 반환."""
    import re
    pattern = re.compile(r"^(.+)_subtask-(\d+|multi_label_classification)_(train|val|test)$")
    results = defaultdict(lambda: {"train": 0, "val": 0, "test": 0})

    if not os.path.isdir(per_task_root):
        return {}

    for entry in sorted(os.listdir(per_task_root)):
        m = pattern.match(entry)
        if not m:
            continue
        task_name, subtask_idx, split = m.groups()
        count = count_arrow_rows(os.path.join(per_task_root, entry))
        if count is not None:
            results[task_name][split] = count
    return dict(results)


print("Helpers loaded.")


## 1. Source Dataset 개요

MolDA 데이터셋은 4개의 주요 소스에서 수집됩니다:

| Source | HuggingFace ID / Method | 논문 | Task 유형 |
|--------|------------------------|------|----------|
| **SMolInstruct** | `osunlp/SMolInstruct` | Yu et al. (2024) | forward synthesis, retrosynthesis, captioning, generation, property prediction, name conversion |
| **Mol-Instructions** | `zjunlp/Mol-Instructions` (Molecule-oriented) | Fang et al. (2023) | forward reaction, retrosynthesis, reagent prediction, QM9 property prediction |
| **ChEBI-20** | `liupf/ChEBI-20-MM` | Edwards et al. (2022) | molecule captioning (mol2text), molecule generation (text2mol) |
| **BACE** | Local CSV (`BioT5_bace_*.csv`) | Subramanian et al. (2016) | BACE inhibitor classification |

### Task → Source 매핑

```
SMolInstruct (osunlp/SMolInstruct):
  - smol-forward_synthesis        (forward reaction prediction)
  - smol-retrosynthesis           (retrosynthesis)
  - smol-molecule_captioning      (molecule → description)
  - smol-molecule_generation      (description → molecule)
  - smol-property_prediction-*    (bbbp, clintox, esol, hiv, lipo, sider)
  - smol-name_conversion-i2s      (IUPAC → SMILES)
  - smol-name_conversion-s2i      (SMILES → IUPAC)

Mol-Instructions (zjunlp/Mol-Instructions):
  - forward_reaction_prediction   (reactants → product)
  - retrosynthesis                (product → reactants)
  - reagent_prediction            (reactants + product → reagent)
  - qm9_homo                     (HOMO energy prediction)
  - qm9_lumo                     (LUMO energy prediction)
  - qm9_homo_lumo_gap            (HOMO-LUMO gap prediction)

ChEBI-20 (liupf/ChEBI-20-MM):
  - chebi-20-mol2text             (molecule → text description)
  - chebi-20-text2mol             (text description → molecule)

Local (BACE):
  - bace                          (BACE inhibitor binary classification)
```


In [ ]:
# ---------------------------------------------------------------------------
# Source → Task 매핑 (generator.py의 get_dataset() 기반)
# ---------------------------------------------------------------------------
SOURCE_TASK_MAP = {
    "SMolInstruct\n(osunlp/SMolInstruct)": [
        "smol-forward_synthesis",
        "smol-retrosynthesis",
        "smol-molecule_captioning",
        "smol-molecule_generation",
        "smol-property_prediction-bbbp",
        "smol-property_prediction-clintox",
        "smol-property_prediction-esol",
        "smol-property_prediction-hiv",
        "smol-property_prediction-lipo",
        "smol-property_prediction-sider",
        "smol-name_conversion-i2s",
        "smol-name_conversion-s2i",
    ],
    "Mol-Instructions\n(zjunlp/Mol-Instructions)": [
        "forward_reaction_prediction",
        "retrosynthesis",
        "reagent_prediction",
        "qm9_homo",
        "qm9_lumo",
        "qm9_homo_lumo_gap",
    ],
    "ChEBI-20\n(liupf/ChEBI-20-MM)": [
        "chebi-20-mol2text",
        "chebi-20-text2mol",
    ],
    "BACE\n(Local CSV)": [
        "bace",
    ],
}

# Task → Source 역매핑
TASK_TO_SOURCE = {}
for source, tasks in SOURCE_TASK_MAP.items():
    for t in tasks:
        TASK_TO_SOURCE[t] = source.split("\n")[0]

# Task 카테고리
TASK_CATEGORY = {}

_CLS = ["bace", "smol-property_prediction-bbbp", "smol-property_prediction-clintox",
        "smol-property_prediction-hiv", "smol-property_prediction-sider"]
_REG = ["smol-property_prediction-esol", "smol-property_prediction-lipo",
        "qm9_homo", "qm9_lumo", "qm9_homo_lumo_gap"]
_RXN = ["forward_reaction_prediction", "retrosynthesis", "reagent_prediction",
        "smol-forward_synthesis", "smol-retrosynthesis"]
_M2T = ["chebi-20-mol2text", "smol-molecule_captioning"]
_T2M = ["chebi-20-text2mol", "smol-molecule_generation"]
_NAME = ["smol-name_conversion-i2s", "smol-name_conversion-s2i"]

for t in _CLS: TASK_CATEGORY[t] = "Classification"
for t in _REG: TASK_CATEGORY[t] = "Regression"
for t in _RXN: TASK_CATEGORY[t] = "Reaction"
for t in _M2T: TASK_CATEGORY[t] = "Mol2Text"
for t in _T2M: TASK_CATEGORY[t] = "Text2Mol"
for t in _NAME: TASK_CATEGORY[t] = "Name Conversion"

print(f"Total tasks: {len(TASK_TO_SOURCE)}")
print(f"Categories: {sorted(set(TASK_CATEGORY.values()))}")


## 2. Task별 처리 후 샘플 수 (Step 1 & Step 2)

Step 1 (Download & Process) 완료 후, 그리고 Step 2 (Decontamination) 완료 후의 per-task Arrow 디렉터리에서 실제 샘플 수를 읽습니다.

- **Step 1**: `Raw/{tag}/step1/{task}_subtask-{i}_{split}/`
- **Step 2**: `Raw/{tag}/step2/{task}_subtask-{i}_{split}/`


In [ ]:
# ---------------------------------------------------------------------------
# Per-Task 샘플 카운트 — Step 1 & Step 2
# ---------------------------------------------------------------------------
step1_counts = scan_per_task_counts(STEP1_ROOT)
step2_counts = scan_per_task_counts(STEP2_ROOT)

print(f"Step 1: {len(step1_counts)} tasks scanned from {STEP1_ROOT}")
print(f"Step 2: {len(step2_counts)} tasks scanned from {STEP2_ROOT}")

# 주 분석 대상은 Step 2 (decontamination 이후); 없으면 Step 1로 fallback
task_counts = step2_counts if step2_counts else step1_counts
primary_label = "Step 2 (decontam)" if step2_counts else "Step 1 (pre-decontam)"
print(f"\nPrimary counts source: {primary_label}")

# DataFrame으로 변환
rows = []
for task in sorted(task_counts.keys()):
    counts = task_counts[task]
    total = counts["train"] + counts["val"] + counts["test"]
    rows.append({
        "Task": task,
        "Source": TASK_TO_SOURCE.get(task, "Unknown"),
        "Category": TASK_CATEGORY.get(task, "Unknown"),
        "Train": counts["train"],
        "Val": counts["val"],
        "Test": counts["test"],
        "Total": total,
    })

df_tasks = pd.DataFrame(rows)
df_tasks = df_tasks.sort_values(["Category", "Source", "Task"]).reset_index(drop=True)
for col in ["Train", "Val", "Test", "Total"]:
    df_tasks[col] = df_tasks[col].astype(int)

print(f"\n=== Per-Task Sample Counts ({primary_label}) ===\n")
display(df_tasks.style.format({
    "Train": "{:,}", "Val": "{:,}", "Test": "{:,}", "Total": "{:,}"
}).set_properties(**{"text-align": "right"}, subset=["Train", "Val", "Test", "Total"]))


In [ ]:
# ---------------------------------------------------------------------------
# Step 1 vs Step 2 비교 (decontam으로 몇 개 제거됐는지)
# ---------------------------------------------------------------------------
if step1_counts and step2_counts:
    comp_rows = []
    for task in sorted(set(step1_counts) | set(step2_counts)):
        c1 = step1_counts.get(task, {"train": 0, "val": 0, "test": 0})
        c2 = step2_counts.get(task, {"train": 0, "val": 0, "test": 0})
        for split in ["train", "val", "test"]:
            before = c1.get(split, 0)
            after = c2.get(split, 0)
            removed = before - after
            if before == 0 and after == 0:
                continue
            comp_rows.append({
                "Task": task,
                "Source": TASK_TO_SOURCE.get(task, "Unknown"),
                "Split": split,
                "Step1": before,
                "Step2": after,
                "Removed (Step 2)": removed,
                "Removed %": (removed / before * 100) if before > 0 else 0.0,
            })
    df_s1s2 = pd.DataFrame(comp_rows)
    df_s1s2_nonzero = df_s1s2[df_s1s2["Removed (Step 2)"] != 0].sort_values(
        "Removed (Step 2)", ascending=False
    ).reset_index(drop=True)

    print("=== Step 2 (Decontamination) 제거 현황 (Step 1 → Step 2) ===\n")
    if len(df_s1s2_nonzero) == 0:
        print("  제거된 샘플 없음 (eval leakage 0, within-family dedup 0)")
    else:
        display(df_s1s2_nonzero.style.format({
            "Step1": "{:,}", "Step2": "{:,}",
            "Removed (Step 2)": "{:,}", "Removed %": "{:.3f}%",
        }))
else:
    print("[Skip] Step 1 또는 Step 2 데이터가 없어 비교 불가")


In [ ]:
# ---------------------------------------------------------------------------
# Source별 집계
# ---------------------------------------------------------------------------
df_source = df_tasks.groupby("Source")[["Train", "Val", "Test", "Total"]].sum()
df_source["# Tasks"] = df_tasks.groupby("Source")["Task"].count()
df_source = df_source[["# Tasks", "Train", "Val", "Test", "Total"]]

print("=== Source별 집계 ===\n")
display(df_source.style.format({
    "# Tasks": "{:d}",
    "Train": "{:,}", "Val": "{:,}", "Test": "{:,}", "Total": "{:,}"
}))

total_train = df_tasks["Train"].sum()
total_val = df_tasks["Val"].sum()
total_test = df_tasks["Test"].sum()
total_all = df_tasks["Total"].sum()
print(f"\n전체 합계: Train={total_train:,} | Val={total_val:,} | Test={total_test:,} | Total={total_all:,}")


In [ ]:
# ---------------------------------------------------------------------------
# Category별 집계
# ---------------------------------------------------------------------------
df_cat = df_tasks.groupby("Category")[["Train", "Val", "Test", "Total"]].sum()
df_cat["# Tasks"] = df_tasks.groupby("Category")["Task"].count()
df_cat = df_cat[["# Tasks", "Train", "Val", "Test", "Total"]]

print("=== Category별 집계 ===\n")
display(df_cat.style.format({
    "# Tasks": "{:d}",
    "Train": "{:,}", "Val": "{:,}", "Test": "{:,}", "Total": "{:,}"
}))


## 3. 최종 합산 데이터셋 (Concatenated Final)

Step 3 (Concatenate & Map) 이후의 최종 Train/Val/Test 데이터셋 크기입니다.

현재 파이프라인은 **하나의 Processed 디렉터리**에 SMILES·SELFIES 듀얼 컬럼으로 저장합니다
(`input_mol_string_smiles`, `prompt_text_smiles`, `target_text_smiles` + 동일한 `_selfies` 버전).
따라서 `Train/Val/Test` 각 split은 SMILES 기준 행 수와 SELFIES 기준 행 수가 동일합니다.


In [ ]:
# ---------------------------------------------------------------------------
# 최종 합산 데이터셋 카운트
# ---------------------------------------------------------------------------
final_rows = []
for split in ["Train", "Val", "Test"]:
    split_dir = os.path.join(PROCESSED_ROOT, split)
    count = count_arrow_rows(split_dir)
    final_rows.append({
        "Split": split,
        "Samples": count if count is not None else 0,
        "Path": split_dir,
    })

df_final = pd.DataFrame(final_rows)
df_final_total = df_final["Samples"].sum()

print(f"=== 최종 데이터셋 (Concatenated, dual SMILES+SELFIES columns) — {DATA_TAG} ===\n")
display(df_final.style.format({"Samples": "{:,}"}))
print(f"\nTotal: {df_final_total:,}")

# per-task 합계 vs final 합계 비교
per_task_total = df_tasks["Total"].sum()
diff = per_task_total - df_final_total
print(f"\n[{primary_label}] Per-task 합계: {per_task_total:,}")
print(f"[Processed Final]   합계: {df_final_total:,}")
print(f"[Diff]              : {diff:,}")
if diff != 0:
    print("  → 차이 원인 후보: Step 3에서 SELFIES 변환 실패 sample 필터 "
          "(`_selfies_valid=False` 제거)")


## 4. 원본 Source Dataset 크기 vs 처리 후 크기 — Preprocessing Rejection 분석

각 소스 데이터셋의 **원본 크기**와 파이프라인 처리 후 남은 크기를 비교합니다.

### Rejection이 발생하는 단계:

| 단계 | 원인 | 해당 코드 |
|------|------|----------|
| **Step 1: SMILES 파싱 실패** | RDKit `MolFromSmiles()` 실패 → `None` 반환 → 해당 sample 제거 | `generator._process_*_sample()` → `return None` |
| **Step 1: SELFIES→SMILES 디코딩 실패** | Mol-Instructions/ChEBI (SELFIES 포맷) | `_selfies_field_to_smiles()`, `_process_chebi_sample()` |
| **Step 1: Graph 변환 실패** | `smiles2data()` (OGB) 실패 → exception | `_process_*_sample()` try/except |
| **Step 1: NaN/None 데이터** | 원본 데이터에 결측값 | `pd.isna()` 체크 |
| **Step 1: Arrow 직렬화 실패** | `dataset_to_arrow_dicts()`에서 `continue` | `generator.dataset_to_arrow_dicts()` |
| **Step 2: Eval Leakage** | Cross-source test/val entity가 train에 존재 → train에서 제거 | `dedup.remove_eval_leakage()` |
| **Step 2: Within-Family Dedup** | 같은 Entity Family 내 cross-source 중복 → priority rule로 제거 | `dedup.dedup_within_family()` |
| **Step 3: SELFIES 변환 실패** | prompt/target의 `<SMILES>` 태그를 `<SELFIES>`로 재변환 실패 (`strict=True`) | `generator._convert_smiles_tags_to_selfies()` + `prepare_data_instance()` |

아래 셀에서는 원본 소스 데이터를 직접 로드하여 처리 전/후 크기를 비교합니다.


In [ ]:
# ---------------------------------------------------------------------------
# 원본 소스 데이터셋 크기 (논문/문서 기반 참조 값)
# ---------------------------------------------------------------------------
KNOWN_SOURCE_SIZES = {
    "SMolInstruct": {
        "train": 3_369_270, "validation": 16_920, "test": 33_747,
        "note": "SMolInstruct paper Table 1 기준 (전체 task 합산, 14 task types)",
    },
    "Mol-Instructions": {
        "forward_reaction_prediction": {"train": 109_485, "test": 27_371},
        "retrosynthesis": {"train": 109_485, "test": 27_371},
        "reagent_prediction": {"train": 109_485, "test": 27_371},
        "property_prediction": {"total": 132_908},
        "note": "zjunlp/Mol-Instructions: Molecule-oriented Instructions subset",
    },
    "ChEBI-20": {
        "train": 26_407, "validation": 3_301, "test": 3_300,
        "note": "liupf/ChEBI-20-MM — mol2text와 text2mol에 동일 원본 사용",
    },
    "BACE": {
        "train": 1_210, "validation": 151, "test": 152,
        "note": "BioT5 BACE local CSV (scaffold split)",
    },
}

print("=== Known Source Dataset Sizes (논문/문서 기준) ===\n")
for source, info in KNOWN_SOURCE_SIZES.items():
    note = info.get("note", "")
    print(f"[{source}] {note}")
    for k, v in info.items():
        if k == "note":
            continue
        if isinstance(v, dict):
            for kk, vv in v.items():
                print(f"  {k}/{kk}: {vv:,}")
        else:
            print(f"  {k}: {v:,}")
    print()


In [ ]:
# ---------------------------------------------------------------------------
# 원본 소스에서 실제 로드하여 per-task 원본 크기 확인
# (HF 캐시가 있으면 빠름. 없으면 LOAD_FROM_SOURCE=False로 skip 가능)
#
# NOTE: generator.py의 get_dataset()과 동일한 로드 기준이되, count용으로는
# ds["task"] / ds["metadata"]를 한 번만 읽어 Counter로 집계한다
# (ds.filter를 task 수만큼 반복하면 각 pass마다 full scan이 일어나 느림).
# ---------------------------------------------------------------------------
import sys, traceback
from collections import Counter

sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

LOAD_FROM_SOURCE = True  # False로 바꾸면 known values만 사용
raw_source_counts = {}

# 원본 HF 객체는 이후 셀(Cell 17, 19)에서도 재사용되므로 유지한다.
smol = None
chebi = None
mol_inst = None

if LOAD_FROM_SOURCE:
    try:
        from datasets import load_dataset

        # --- SMolInstruct ---
        # 3.3M rows × 14 tasks를 filter로 돌리면 ~15s × 14 = 3분 이상 소요됨.
        # task 컬럼만 한 번 읽어 Counter로 집계하면 1회 스캔으로 끝남.
        print("[Loading] SMolInstruct...")
        smol = load_dataset(
            "osunlp/SMolInstruct",
            use_selfies=False,
            insert_core_tags=False,
            trust_remote_code=True,
        )
        for split_name, split_ds in smol.items():
            task_counter = Counter(split_ds["task"])
            for task, n in task_counter.items():
                key = f"smol-{task}"
                raw_source_counts.setdefault(key, {})[split_name] = n
        print(f"  SMolInstruct loaded: train={len(smol['train'])}, "
              f"val={len(smol.get('validation', []))}, test={len(smol['test'])}")

        # --- ChEBI-20 ---
        print("[Loading] ChEBI-20...")
        chebi = load_dataset("liupf/ChEBI-20-MM")
        for task in ["chebi-20-mol2text", "chebi-20-text2mol"]:
            raw_source_counts[task] = {
                "train": len(chebi["train"]),
                "validation": len(chebi["validation"]),
                "test": len(chebi["test"]),
            }
        print(f"  ChEBI-20 loaded: train={len(chebi['train'])}")

        # --- Mol-Instructions ---
        # metadata 컬럼 한 번 읽어 split별 count 집계 (filter 2회 스캔 회피).
        print("[Loading] Mol-Instructions...")
        mol_inst = load_dataset(
            "zjunlp/Mol-Instructions",
            "Molecule-oriented Instructions",
            trust_remote_code=True,
        )

        def _split_counts(ds):
            train_c, test_c = 0, 0
            for meta in ds["metadata"]:
                if "train" in meta:
                    train_c += 1
                elif "test" in meta:
                    test_c += 1
            return train_c, test_c

        for task_key in ["forward_reaction_prediction", "retrosynthesis", "reagent_prediction"]:
            if task_key not in mol_inst:
                continue
            ds = mol_inst[task_key]
            train_c, test_c = _split_counts(ds)
            raw_source_counts[task_key] = {
                "train": train_c, "test": test_c, "total": len(ds),
            }

        # QM9 property prediction: instruction → (qm9_task, metadata split) 1-pass 집계
        if "property_prediction" in mol_inst:
            import instructions_smol
            pp_ds = mol_inst["property_prediction"]

            # instruction → qm9_task 역매핑 (한 번만 빌드)
            instr_to_task = {}
            for qm9_task, subtask_name in [
                ("qm9_homo", "homo"),
                ("qm9_lumo", "lumo"),
                ("qm9_homo_lumo_gap", "homo_lumo_gap"),
            ]:
                for t in getattr(instructions_smol, f"filtering_template_{subtask_name}", []):
                    instr_to_task[t] = qm9_task
                raw_source_counts[qm9_task] = {"train": 0, "test": 0, "total": 0}

            # 단일 pass: 두 컬럼을 함께 순회
            for instr, meta in zip(pp_ds["instruction"], pp_ds["metadata"]):
                qm9_task = instr_to_task.get(instr)
                if qm9_task is None:
                    continue
                raw_source_counts[qm9_task]["total"] += 1
                if "train" in meta:
                    raw_source_counts[qm9_task]["train"] += 1
                elif "test" in meta:
                    raw_source_counts[qm9_task]["test"] += 1
        print("  Mol-Instructions loaded")

        # --- BACE (local CSV) ---
        bace_root = os.path.join(RAW_DATA_ROOT, "raw")
        for split, fname in [("train", "BioT5_bace_train.csv"),
                              ("validation", "BioT5_bace_valid.csv"),
                              ("test", "BioT5_bace_test.csv")]:
            fpath = os.path.join(bace_root, fname)
            if os.path.exists(fpath):
                df = pd.read_csv(fpath)
                raw_source_counts.setdefault("bace", {})[split] = len(df)
        print("  BACE loaded")

        print(f"\n총 {len(raw_source_counts)} tasks의 원본 크기를 로드했습니다.")

    except Exception as e:
        print(f"[Warning] Source loading failed: {e}")
        traceback.print_exc()
        print("Known values만 사용합니다.")
        LOAD_FROM_SOURCE = False


In [ ]:
# ---------------------------------------------------------------------------
# Rejection 분석: 원본 → Step 2 최종 비교
# ---------------------------------------------------------------------------
# NOTE: Mol-Instructions 계열은 train에서 2%를 val로 split하므로
#       raw train → processed train에는 해당 2%만큼 차이가 기본 발생.
rejection_rows = []

for task in sorted(task_counts.keys()):
    processed = task_counts[task]
    raw = raw_source_counts.get(task, {})

    for split in ["train", "val", "test"]:
        processed_count = processed.get(split, 0)
        raw_split = split if split != "val" else "validation"
        raw_count = raw.get(raw_split)

        # Mol-Instructions tasks: val은 train에서 2% split되므로 raw val이 없음
        if raw_count is None and split == "val":
            if task in ["forward_reaction_prediction", "retrosynthesis", "reagent_prediction",
                        "qm9_homo", "qm9_lumo", "qm9_homo_lumo_gap"]:
                raw_train = raw.get("train", 0)
                if raw_train > 0:
                    raw_count = int(raw_train * 0.02)

        if raw_count is None:
            raw_count_str = "N/A"; rejected = None; rejection_rate = None
        else:
            raw_count_str = raw_count
            rejected = raw_count - processed_count
            rejection_rate = (rejected / raw_count * 100) if raw_count > 0 else 0.0

        rejection_rows.append({
            "Task": task,
            "Source": TASK_TO_SOURCE.get(task, "Unknown"),
            "Split": split,
            "Raw (Source)": raw_count_str,
            "After Pipeline": processed_count,
            "Rejected": rejected if rejected is not None else "N/A",
            "Rejection %": rejection_rate if rejection_rate is not None else None,
        })

df_rejection = pd.DataFrame(rejection_rows)

# 유효한 rejection이 있는 행만
df_rej_valid = df_rejection[df_rejection["Rejected"] != "N/A"].copy()
df_rej_valid["Raw (Source)"] = pd.to_numeric(df_rej_valid["Raw (Source)"])
df_rej_valid["Rejected"] = pd.to_numeric(df_rej_valid["Rejected"])

print(f"=== Preprocessing Rejection 분석 (원본 → {primary_label}) ===\n")
print("NOTE: Rejected = Raw(Source) - After Pipeline")
print("  - Step 1: SMILES 파싱/SELFIES 디코딩/Graph 변환 실패, NaN 데이터")
print("  - Step 2: Eval leakage + Within-family dedup (reaction families only)")
print()

df_rej_train = df_rej_valid[df_rej_valid["Split"] == "train"].copy()
if len(df_rej_train) > 0:
    print("--- Train Split ---\n")
    display(df_rej_train[["Task", "Source", "Raw (Source)", "After Pipeline", "Rejected", "Rejection %"]]
            .sort_values("Rejected", ascending=False)
            .reset_index(drop=True)
            .style.format({
                "Raw (Source)": "{:,.0f}",
                "After Pipeline": "{:,}",
                "Rejected": "{:,.0f}",
                "Rejection %": "{:.2f}%",
            }))

df_rej_test = df_rej_valid[df_rej_valid["Split"] == "test"].copy()
if len(df_rej_test) > 0:
    print("\n--- Test Split ---\n")
    display(df_rej_test[["Task", "Source", "Raw (Source)", "After Pipeline", "Rejected", "Rejection %"]]
            .sort_values("Rejected", ascending=False)
            .reset_index(drop=True)
            .style.format({
                "Raw (Source)": "{:,.0f}",
                "After Pipeline": "{:,}",
                "Rejected": "{:,.0f}",
                "Rejection %": "{:.2f}%",
            }))


### 4.1 Rejected Sample 구체적 예시 — Step 1: Preprocessing

Step 1(`generator.py`의 `_process_*_sample()` 함수들)에서 sample이 reject되는 주요 사유:

| # | Rejection 사유 | 해당 소스 | 상세 |
|---|----------------|-----------|------|
| 1 | **SMILES 파싱 실패** | 모든 소스 | `get_canonical_smiles(smiles)` → `None` (RDKit `MolFromSmiles` 실패) |
| 2 | **SELFIES→SMILES 디코딩 실패** | Mol-Instructions (항상 SELFIES), ChEBI (SELFIES 컬럼일 때), BACE (SELFIES 컬럼) | `selfies_to_smiles()` 또는 `_selfies_field_to_smiles()` → `None` |
| 3 | **Graph 변환 실패** | 모든 소스 | `smiles2data()` (OGB `smiles2graph`) 내부 exception |
| 4 | **NaN/None 데이터** | Mol-Instructions, ChEBI | `pd.isna(input)` 또는 `pd.isna(output)` → raise → `except` |

이 네 가지는 모두 `_process_*_sample()` 내부 try/except에서 잡혀서 `return None`을 반환하고,
이후 `valid = [r for r in results if r is not None]` 필터에서 제외됩니다.


In [ ]:
# ---------------------------------------------------------------------------
# Step 1 Rejection 예시: 실제 raw 데이터에서 rejected sample 찾기
# ---------------------------------------------------------------------------
import sys
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

from dataset_generation.utils import get_canonical_smiles, selfies_to_smiles, clean_mol_string

MAX_EXAMPLES = 5
SCAN_LIMIT = 2000

rejection_examples = defaultdict(list)

# ── 1. SMolInstruct: SMILES 파싱 실패 ──
if "smol" in dir():
    print("[Scanning] SMolInstruct — SMILES 파싱 실패 탐색...")
    smol_tasks = [
        "forward_synthesis", "retrosynthesis", "molecule_captioning",
        "molecule_generation", "property_prediction-bbbp", "property_prediction-clintox",
        "property_prediction-esol", "property_prediction-hiv", "property_prediction-lipo",
        "property_prediction-sider", "name_conversion-i2s", "name_conversion-s2i",
    ]
    for smol_task in smol_tasks:
        if len(rejection_examples["smiles_parse_fail"]) >= MAX_EXAMPLES:
            break
        task_ds = smol["train"].filter(lambda x, t=smol_task: x["task"] == t)
        for i in range(min(SCAN_LIMIT, len(task_ds))):
            row = task_ds[i]
            raw_input = str(row["raw_input"])
            # text2mol/i2s는 텍스트 입력 → SMILES 파싱 대상 아님
            if smol_task in ["molecule_generation", "name_conversion-i2s"]:
                continue
            cleaned = clean_mol_string(raw_input)
            canonical = get_canonical_smiles(cleaned)
            if canonical is None and cleaned and ">>" not in cleaned:
                rejection_examples["smiles_parse_fail"].append({
                    "task": f"smol-{smol_task}",
                    "raw_value": raw_input[:120],
                    "reason_detail": f"get_canonical_smiles('{cleaned[:60]}...') → None  (RDKit 파싱 실패)",
                })
                if len(rejection_examples["smiles_parse_fail"]) >= MAX_EXAMPLES:
                    break
else:
    print("[Skip] SMolInstruct 미로드 — Cell 14를 먼저 실행하세요.")

# ── 2. Mol-Instructions: SELFIES→SMILES 변환 실패 ──
if "mol_inst" in dir():
    print("[Scanning] Mol-Instructions — SELFIES→SMILES 변환 실패 탐색...")
    for task_key in ["forward_reaction_prediction", "retrosynthesis", "reagent_prediction"]:
        if len(rejection_examples["selfies_decode_fail"]) >= MAX_EXAMPLES:
            break
        if task_key not in mol_inst:
            continue
        ds = mol_inst[task_key]
        train_ds = ds.filter(lambda x: "train" in x["metadata"])
        for i in range(min(SCAN_LIMIT, len(train_ds))):
            row = train_ds[i]
            input_val = str(row["input"])
            if pd.isna(row["input"]):
                rejection_examples["nan_data"].append({
                    "task": task_key, "raw_value": "(NaN)",
                    "reason_detail": "input 필드가 NaN → pd.isna() 체크에서 reject",
                })
                continue
            if ">>" in input_val:
                parts = input_val.split(">>")
                for p in parts:
                    converted = selfies_to_smiles(p.strip())
                    if converted is None and p.strip():
                        rejection_examples["selfies_decode_fail"].append({
                            "task": task_key,
                            "raw_value": input_val[:120],
                            "reason_detail": f"selfies_to_smiles('{p.strip()[:50]}...') → None",
                        })
                        break
            else:
                converted = selfies_to_smiles(input_val)
                if converted is None:
                    rejection_examples["selfies_decode_fail"].append({
                        "task": task_key,
                        "raw_value": input_val[:120],
                        "reason_detail": f"selfies_to_smiles() → None",
                    })
            if len(rejection_examples["selfies_decode_fail"]) >= MAX_EXAMPLES:
                break

    # QM9 property prediction: NaN 체크
    if "property_prediction" in mol_inst:
        pp_ds = mol_inst["property_prediction"]
        for i in range(min(SCAN_LIMIT, len(pp_ds))):
            if len(rejection_examples["nan_data"]) >= MAX_EXAMPLES:
                break
            row = pp_ds[i]
            if pd.isna(row["input"]) or pd.isna(row["output"]):
                rejection_examples["nan_data"].append({
                    "task": "property_prediction",
                    "raw_value": f"input={str(row['input'])[:40]}, output={str(row['output'])[:40]}",
                    "reason_detail": "input 또는 output이 NaN",
                })
else:
    print("[Skip] Mol-Instructions 미로드 — Cell 14를 먼저 실행하세요.")

# ── 3. ChEBI-20: SELFIES/SMILES 변환 실패 ──
if "chebi" in dir():
    print("[Scanning] ChEBI-20 — 변환 실패 탐색...")
    chebi_train = chebi["train"]
    cols = chebi_train.column_names
    selfies_col = "SELFIES" if "SELFIES" in cols else None
    smiles_col = "SMILES" if "SMILES" in cols else ("smiles" if "smiles" in cols else None)
    desc_col = "description" if "description" in cols else ("text" if "text" in cols else None)

    mol_col = selfies_col or smiles_col
    if mol_col:
        for i in range(min(SCAN_LIMIT, len(chebi_train))):
            if len(rejection_examples.get("chebi_fail", [])) >= MAX_EXAMPLES:
                break
            row = chebi_train[i]
            mol_data = row[mol_col]
            desc = row[desc_col] if desc_col else None
            if pd.isna(mol_data) or pd.isna(desc):
                rejection_examples["nan_data"].append({
                    "task": "chebi-20-mol2text",
                    "raw_value": f"mol={str(mol_data)[:40]}, desc={str(desc)[:40]}",
                    "reason_detail": "description 또는 molecule 데이터가 NaN",
                })
                continue
            if selfies_col:
                smiles = selfies_to_smiles(str(mol_data))
                if smiles is None:
                    rejection_examples["selfies_decode_fail"].append({
                        "task": "chebi-20-mol2text",
                        "raw_value": str(mol_data)[:120],
                        "reason_detail": f"selfies_to_smiles('{str(mol_data)[:50]}...') → None",
                    })
            else:
                canonical = get_canonical_smiles(str(mol_data))
                if canonical is None:
                    rejection_examples["smiles_parse_fail"].append({
                        "task": "chebi-20-mol2text",
                        "raw_value": str(mol_data)[:120],
                        "reason_detail": f"get_canonical_smiles() → None",
                    })
else:
    print("[Skip] ChEBI-20 미로드 — Cell 14를 먼저 실행하세요.")

# ── 4. BACE: SELFIES→SMILES 변환 실패 ──
bace_path = os.path.join(RAW_DATA_ROOT, "raw", "BioT5_bace_train.csv")
if os.path.exists(bace_path):
    print("[Scanning] BACE — SELFIES 변환 실패 탐색...")
    bace_df = pd.read_csv(bace_path)
    selfies_col_bace = "SELFIES" if "SELFIES" in bace_df.columns else None
    if selfies_col_bace:
        for i in range(min(SCAN_LIMIT, len(bace_df))):
            if len(rejection_examples["selfies_decode_fail"]) >= MAX_EXAMPLES:
                break
            val = bace_df.iloc[i][selfies_col_bace]
            if pd.isna(val):
                rejection_examples["nan_data"].append({
                    "task": "bace", "raw_value": "(NaN SELFIES)",
                    "reason_detail": "SELFIES 컬럼이 NaN",
                })
                continue
            smiles = selfies_to_smiles(str(val))
            if smiles is None:
                rejection_examples["selfies_decode_fail"].append({
                    "task": "bace",
                    "raw_value": str(val)[:120],
                    "reason_detail": f"selfies_to_smiles() → None",
                })

# ── 결과 출력 ──
reason_labels = {
    "smiles_parse_fail": "SMILES 파싱 실패 (RDKit MolFromSmiles → None)",
    "selfies_decode_fail": "SELFIES→SMILES 변환 실패 (selfies_to_smiles → None)",
    "nan_data": "NaN/None 데이터 (input 또는 label이 NaN)",
}

total_found = sum(len(v) for v in rejection_examples.values())
print(f"\n{'='*80}")
print(f"Step 1 Rejection 예시 총 {total_found}개 발견")
print(f"{'='*80}")

for reason, label in reason_labels.items():
    examples = rejection_examples.get(reason, [])
    print(f"\n{'─'*80}")
    print(f"▶ {label}")
    print(f"{'─'*80}")
    if not examples:
        print("  (스캔 범위 내에서 해당 예시 없음)")
        continue
    for j, ex in enumerate(examples[:MAX_EXAMPLES]):
        print(f"\n  [{j+1}] Task: {ex['task']}")
        print(f"      Raw value: {ex['raw_value']}")
        print(f"      → {ex['reason_detail']}")
        print(f"      ∴ _process_*_sample()에서 return None → 최종 데이터셋에서 제외됨")


### 4.2 Rejected Sample 구체적 예시 — Step 2: Cross-Source Decontamination

Step 2에서는 **서로 다른 소스 간 분자/반응 중복**을 제거합니다. 두 가지 메커니즘이 있습니다:

#### Step 2a — Eval Blacklist 수집
`dedup.build_eval_blacklist()` — test+val의 entity key를 family별 blacklist에 수집 (제거는 안 함).

#### Step 2b — Eval Leakage Removal (`dedup.remove_eval_leakage`)
Family 내에서 **train** 샘플의 entity key가 eval blacklist에 있으면 train에서 제거.
(val은 eval boundary의 일부로 간주되어 건드리지 않음)

#### Step 2c — Within-Family Dedup (`dedup.dedup_within_family`)
같은 family에 속한 task들 간 train 중복을 **`REMOVE_ON_CONFLICT`에 지정된 task**에서 제거.
현재 코드에서는 reaction 두 family만 대상 — MOL2TEXT/TEXT2MOL family는 의도적으로 제외됩니다:

| Family | Priority (anchor) | Removed on conflict |
|--------|-------------------|---------------------|
| `REACTION_FORWARD_FAMILY` | `smol-forward_synthesis` | `forward_reaction_prediction` |
| `REACTION_RETRO_FAMILY` | `smol-retrosynthesis` | `retrosynthesis` |
| `MOL2TEXT_FAMILY` | (양쪽 다 보존) | — |
| `TEXT2MOL_FAMILY` | (양쪽 다 보존) | — |

Entity key 추출 컬럼은 family마다 다름 (`FAMILY_KEY_SOURCE`):
- `TEXT2MOL_FAMILY` → `label` 컬럼 (input_mol_string은 `<None>`이기 때문)
- 그 외 → `input_mol_string` 컬럼


In [ ]:
# ---------------------------------------------------------------------------
# Step 2 Rejection 예시: Eval Leakage + Within-Family Dedup 시뮬레이션
# ---------------------------------------------------------------------------
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

import datasets as hf_datasets
from dataset_generation.dedup import (
    ENTITY_FAMILIES, REMOVE_ON_CONFLICT, FAMILY_KEY_SOURCE,
)
from dataset_generation.utils import extract_entity_key

MAX_EXAMPLES = 5

# Family별 task 목록
family_tasks_map = defaultdict(list)
for task, family in ENTITY_FAMILIES.items():
    family_tasks_map[family].append(task)

def _key_field_for_family(family):
    return FAMILY_KEY_SOURCE.get(family, "input_mol_string")

# 분석 대상은 Step 1 결과 (Step 2로 들어가기 직전). Step 1이 없으면 Step 2 fallback.
eval_scan_root = STEP1_ROOT if os.path.isdir(STEP1_ROOT) else STEP2_ROOT

# =========================================================================
# Part A: Eval Leakage 시뮬레이션
# =========================================================================
print("=" * 80)
print("▶ Step 2b: Eval Leakage 제거 예시")
print("  (Step 1 test/val의 entity key가 raw train에도 존재하는 경우)")
print("=" * 80)

leakage_examples = []

for family, tasks in family_tasks_map.items():
    if len(leakage_examples) >= MAX_EXAMPLES:
        break

    key_field = _key_field_for_family(family)

    # eval entity key 수집 (family 단위 blacklist 재현)
    eval_keys = {}
    for task in tasks:
        for split in ["test", "val"]:
            path = os.path.join(eval_scan_root, f"{task}_subtask-0_{split}")
            if not os.path.isdir(path):
                continue
            try:
                ds = hf_datasets.Dataset.load_from_disk(path)
                if key_field not in ds.column_names:
                    continue
                for mol_str in ds[key_field]:
                    key = extract_entity_key(mol_str)
                    if key and key not in eval_keys:
                        eval_keys[key] = (task, split, str(mol_str)[:80])
            except Exception:
                continue

    if not eval_keys:
        continue

    # raw HF train에서 eval key와 겹치는 sample 찾기
    # Mol-Instructions 계열
    if "mol_inst" in dir():
        for task in tasks:
            if len(leakage_examples) >= MAX_EXAMPLES:
                break
            if task not in ["forward_reaction_prediction", "retrosynthesis", "reagent_prediction"]:
                continue
            if task not in mol_inst:
                continue
            raw_ds = mol_inst[task]
            raw_train = raw_ds.filter(lambda x: "train" in x["metadata"])
            for i in range(min(SCAN_LIMIT, len(raw_train))):
                row = raw_train[i]
                input_val = str(row["input"])
                if ">>" in input_val:
                    parts = input_val.split(">>")
                    converted = []
                    for p in parts:
                        s = selfies_to_smiles(p.strip())
                        if s is None:
                            break
                        converted.append(s)
                    if len(converted) == len(parts):
                        key = extract_entity_key(">>".join(converted))
                    else:
                        continue
                else:
                    smiles_str = selfies_to_smiles(input_val)
                    if smiles_str is None:
                        continue
                    key = extract_entity_key(smiles_str)

                if key and key in eval_keys:
                    eval_task, eval_split, eval_mol = eval_keys[key]
                    leakage_examples.append({
                        "family": family,
                        "train_task": task,
                        "train_raw": input_val[:80],
                        "eval_task": eval_task,
                        "eval_split": eval_split,
                        "entity_key": key[:80],
                    })
                    if len(leakage_examples) >= MAX_EXAMPLES:
                        break

    # ChEBI-20 계열 (mol2text=input 분자, text2mol=label 분자)
    if "chebi" in dir():
        for task in tasks:
            if len(leakage_examples) >= MAX_EXAMPLES:
                break
            if task not in ["chebi-20-mol2text", "chebi-20-text2mol"]:
                continue
            chebi_train = chebi["train"]
            cols = chebi_train.column_names
            smiles_col = "SMILES" if "SMILES" in cols else ("smiles" if "smiles" in cols else None)
            if not smiles_col:
                continue
            for i in range(min(SCAN_LIMIT, len(chebi_train))):
                mol_str = str(chebi_train[i][smiles_col])
                key = extract_entity_key(mol_str)
                if key and key in eval_keys:
                    eval_task, eval_split, eval_mol = eval_keys[key]
                    leakage_examples.append({
                        "family": family,
                        "train_task": task,
                        "train_raw": mol_str[:80],
                        "eval_task": eval_task,
                        "eval_split": eval_split,
                        "entity_key": key[:80],
                    })
                    if len(leakage_examples) >= MAX_EXAMPLES:
                        break

if leakage_examples:
    for j, ex in enumerate(leakage_examples[:MAX_EXAMPLES]):
        print(f"\n  [{j+1}] Family: {ex['family']}")
        print(f"      {ex['train_task']} train의 raw input: '{ex['train_raw']}...'")
        print(f"      Entity key: '{ex['entity_key']}'")
        print(f"      → {ex['eval_task']} {ex['eval_split']}에 이미 존재")
        print(f"      ∴ eval leakage로 train에서 제거됨")
else:
    print("\n  (스캔 범위 내에서 eval leakage 예시를 찾지 못함)")
    print("  → Mol-Instructions/ChEBI train과 SMolInstruct test 간 겹치는 entity가")
    print(f"    스캔 범위({SCAN_LIMIT}) 내에 없었을 가능성이 높음.")

# =========================================================================
# Part B: Within-Family Dedup 시뮬레이션 (reaction families only)
# =========================================================================
print(f"\n{'='*80}")
print("▶ Step 2c: Within-Family Dedup 예시")
print("  (anchor task train의 entity key가 remove task train에도 존재)")
print("  NOTE: MOL2TEXT/TEXT2MOL family는 의도적으로 제외됨.")
print("=" * 80)

dedup_examples = []

for family, tasks in family_tasks_map.items():
    if len(dedup_examples) >= MAX_EXAMPLES:
        break

    remove_task = REMOVE_ON_CONFLICT.get(family)
    anchor_tasks = [t for t in tasks if t != remove_task]
    if not remove_task or not anchor_tasks:
        continue

    key_field = _key_field_for_family(family)

    print(f"\n  [{family}] (key_field={key_field})")
    print(f"    Anchor (보존): {anchor_tasks}")
    print(f"    Remove (충돌 시 제거): {remove_task}")

    # anchor task train의 entity key 수집
    anchor_keys = {}
    for at in anchor_tasks:
        path = os.path.join(eval_scan_root, f"{at}_subtask-0_train")
        if not os.path.isdir(path):
            continue
        try:
            ds = hf_datasets.Dataset.load_from_disk(path)
            if key_field not in ds.column_names:
                continue
            for mol_str in ds[key_field]:
                key = extract_entity_key(mol_str)
                if key and key not in anchor_keys:
                    anchor_keys[key] = (at, str(mol_str)[:80])
        except Exception:
            continue

    if not anchor_keys:
        print("    (anchor train 데이터 없음)")
        continue

    # remove_task raw train에서 anchor key와 겹치는 sample 찾기
    if remove_task in ["forward_reaction_prediction", "retrosynthesis", "reagent_prediction"]:
        if "mol_inst" in dir() and remove_task in mol_inst:
            raw_ds = mol_inst[remove_task]
            raw_train = raw_ds.filter(lambda x: "train" in x["metadata"])
            for i in range(min(SCAN_LIMIT, len(raw_train))):
                row = raw_train[i]
                input_val = str(row["input"])
                if ">>" in input_val:
                    parts = input_val.split(">>")
                    converted = []
                    for p in parts:
                        s = selfies_to_smiles(p.strip())
                        if s is None:
                            break
                        converted.append(s)
                    if len(converted) == len(parts):
                        key = extract_entity_key(">>".join(converted))
                    else:
                        continue
                else:
                    smiles_str = selfies_to_smiles(input_val)
                    if smiles_str is None:
                        continue
                    key = extract_entity_key(smiles_str)

                if key and key in anchor_keys:
                    anchor_task, anchor_mol = anchor_keys[key]
                    dedup_examples.append({
                        "family": family,
                        "remove_task": remove_task,
                        "remove_raw": input_val[:80],
                        "anchor_task": anchor_task,
                        "anchor_mol": anchor_mol,
                        "entity_key": key[:80],
                    })
                    if len(dedup_examples) >= MAX_EXAMPLES:
                        break

if dedup_examples:
    print(f"\n{'─'*80}")
    print("  Within-Family Dedup으로 제거된 sample 예시:")
    print(f"{'─'*80}")
    for j, ex in enumerate(dedup_examples[:MAX_EXAMPLES]):
        print(f"\n  [{j+1}] Family: {ex['family']}")
        print(f"      {ex['remove_task']} train의 raw input: '{ex['remove_raw']}...'")
        print(f"      Entity key: '{ex['entity_key']}'")
        print(f"      → {ex['anchor_task']} train에도 동일 entity 존재: '{ex['anchor_mol']}...'")
        print(f"      ∴ {ex['remove_task']}는 REMOVE_ON_CONFLICT 대상 → within_family_dup로 제거")
else:
    print("\n  (스캔 범위 내에서 within-family dedup 예시를 찾지 못함)")

print(f"\n{'='*80}")
print(f"총 leakage 예시: {len(leakage_examples)}개, dedup 예시: {len(dedup_examples)}개")
print(f"{'='*80}")

RDLogger.EnableLog("rdApp.*")


### 4.3 Step 3: Concatenate & Map — SELFIES 변환 실패 필터

Step 3(`generator.prepare_data_instance()`)은 `input_mol_string` / `label`의 `<SMILES>` 태그를
`<SELFIES>`로 재변환하여 `*_smiles` / `*_selfies` 듀얼 컬럼을 생성합니다.

이 과정에서 `_convert_smiles_tags_to_selfies(..., strict=True)`가 실패하면 sample에
`_selfies_valid=False`가 마킹되고, `.map()` 이후 `.filter(lambda x: x["_selfies_valid"])`로 제거됩니다.

즉, Step 3에도 일부 rejection이 존재할 수 있습니다 (주로 SELFIES 인코딩이 어려운 특수 구조).


In [ ]:
# ---------------------------------------------------------------------------
# 전체 Rejection Flow 요약
# ---------------------------------------------------------------------------
print("=" * 80)
print("Pipeline 전체 Rejection Flow 요약")
print("=" * 80)

summary_data = [
    {
        "단계": "Step 1: Preprocessing",
        "위치": "generator.py → _process_*_sample()",
        "Rejection 사유": "SMILES 파싱 / SELFIES 디코딩 / Graph 변환 / NaN",
        "메커니즘": "return None → valid = [r for r in results if r is not None]",
        "영향 범위": "모든 소스 (SMolInstruct, Mol-Instructions, ChEBI-20, BACE)",
    },
    {
        "단계": "Step 1: Arrow 직렬화",
        "위치": "generator.py → dataset_to_arrow_dicts()",
        "Rejection 사유": "그래프 tensor 접근 중 예외 (드묾)",
        "메커니즘": "except Exception: continue",
        "영향 범위": "모든 task",
    },
    {
        "단계": "Step 2a: Eval Blacklist",
        "위치": "dedup.py → build_eval_blacklist()",
        "Rejection 사유": "(제거 안 함 — entity key 수집만)",
        "메커니즘": "test+val entity key를 family별 blacklist에 추가",
        "영향 범위": "Entity Family에 속한 task만",
    },
    {
        "단계": "Step 2b: Eval Leakage",
        "위치": "dedup.py → remove_eval_leakage()",
        "Rejection 사유": "train sample의 entity key가 eval blacklist에 존재",
        "메커니즘": "key ∈ blacklist → keep_indices에서 제외",
        "영향 범위": "Family 내 cross-source eval leakage",
    },
    {
        "단계": "Step 2c: Family Dedup",
        "위치": "dedup.py → dedup_within_family()",
        "Rejection 사유": "remove task train의 entity key가 anchor task train에 존재",
        "메커니즘": "REMOVE_ON_CONFLICT에 지정된 task에서 제거",
        "영향 범위": "REACTION_FORWARD_FAMILY, REACTION_RETRO_FAMILY 만 (2개)",
    },
    {
        "단계": "Step 3: SELFIES Map+Filter",
        "위치": "generator.py → prepare_data_instance() + .filter()",
        "Rejection 사유": "`<SMILES>` → `<SELFIES>` 변환 실패 (strict=True)",
        "메커니즘": "_selfies_valid=False → .filter로 제거",
        "영향 범위": "모든 task (주로 특수 구조 분자)",
    },
]

df_pipeline_summary = pd.DataFrame(summary_data)
display(df_pipeline_summary.style.set_properties(**{
    "text-align": "left",
    "white-space": "pre-wrap",
}).set_table_styles([
    {"selector": "th", "props": [("text-align", "center")]},
]))


## 5. Cross-Source Decontamination 통계

Step 2에서 수행되는 cross-source decontamination의 영향을 분석합니다.

### Entity Family 구조

| Family | Tasks | Priority (Anchor) | Removed on Conflict |
|--------|-------|-------------------|---------------------|
| `REACTION_FORWARD_FAMILY` | `forward_reaction_prediction`, `smol-forward_synthesis` | `smol-forward_synthesis` | `forward_reaction_prediction` |
| `REACTION_RETRO_FAMILY`   | `retrosynthesis`, `smol-retrosynthesis` | `smol-retrosynthesis` | `retrosynthesis` |
| `MOL2TEXT_FAMILY`         | `chebi-20-mol2text`, `smol-molecule_captioning` | (양쪽 train 보존) | — |
| `TEXT2MOL_FAMILY`         | `chebi-20-text2mol`, `smol-molecule_generation` | (양쪽 train 보존) | — |

- **Eval blacklist (Step 2a/2b)**: 모든 family에 대해 수행. test+val entity가 train에 있으면 train에서 제거.
- **Within-family dedup (Step 2c)**: Reaction family 2개만 수행. MOL2TEXT/TEXT2MOL은 의도적으로 제외.


In [ ]:
# ---------------------------------------------------------------------------
# Entity Family 정보 출력
# ---------------------------------------------------------------------------
print("=== Entity Families ===\n")
family_tasks = defaultdict(list)
for task, family in ENTITY_FAMILIES.items():
    family_tasks[family].append(task)

for family, tasks in family_tasks.items():
    remove_task = REMOVE_ON_CONFLICT.get(family, "(none — both preserved)")
    anchor_tasks = [t for t in tasks if t != REMOVE_ON_CONFLICT.get(family)]
    key_field = FAMILY_KEY_SOURCE.get(family, "input_mol_string")
    print(f"  {family}:")
    print(f"    Anchor (보존):       {anchor_tasks}")
    print(f"    Remove (충돌 시 제거): {remove_task}")
    print(f"    Key source column:   {key_field}")
    print()

# Family에 속하지 않는 독립 task
independent_tasks = [t for t in task_counts.keys() if t not in ENTITY_FAMILIES]
print(f"독립 Tasks (Family 없음): {len(independent_tasks)}개")
for t in sorted(independent_tasks):
    print(f"  - {t}")


In [ ]:
# ---------------------------------------------------------------------------
# Entity key overlap 분석 (Step 2 결과 기반)
#
# family 내 task 간 entity key overlap을 직접 측정.
# overlap=0이면 dedup이 정상 작동한 것 (reaction family).
# MOL2TEXT/TEXT2MOL은 within-family dedup을 수행하지 않으므로 overlap>0일 수 있음.
# ---------------------------------------------------------------------------
overlap_scan_root = STEP2_ROOT if step2_counts else STEP1_ROOT
print(f"Scanning from: {overlap_scan_root}\n")

def load_entity_keys_for_task(per_task_root, task, split, key_field):
    path = os.path.join(per_task_root, f"{task}_subtask-0_{split}")
    if not os.path.isdir(path):
        return set()
    try:
        ds = hf_datasets.Dataset.load_from_disk(path)
        if key_field not in ds.column_names:
            return set()
        keys = set()
        for mol_str in ds[key_field]:
            key = extract_entity_key(mol_str)
            if key:
                keys.add(key)
        return keys
    except Exception as e:
        print(f"  [Warning] Failed to load {path}: {e}")
        return set()


print("=== Cross-Source Entity Key Overlap 분석 ===\n")
print("(Step 2 이후 데이터 기준)")
print("  - Reaction family: overlap=0 기대 (within_family_dup 제거됨)")
print("  - Mol2Text/Text2Mol family: within dedup 미수행 → overlap>0 가능")
print()

overlap_rows = []
for family, tasks in family_tasks.items():
    if len(tasks) < 2:
        continue

    key_field = FAMILY_KEY_SOURCE.get(family, "input_mol_string")
    print(f"--- {family}  (key_field={key_field}) ---")

    task_keys = {}
    for task in tasks:
        task_keys[task] = {}
        for split in ["train", "test"]:
            keys = load_entity_keys_for_task(overlap_scan_root, task, split, key_field)
            task_keys[task][split] = keys
            print(f"  {task}/{split}: {len(keys):,} unique entity keys")

    for i, t1 in enumerate(tasks):
        for t2 in tasks[i+1:]:
            for (split_a, split_b) in [("train", "test"), ("test", "train")]:
                overlap = task_keys[t1][split_a] & task_keys[t2][split_b]
                status = "CLEAN" if len(overlap) == 0 else "LEAKAGE!"
                overlap_rows.append({
                    "Family": family,
                    "Task A": f"{t1}/{split_a}",
                    "Task B": f"{t2}/{split_b}",
                    "Overlap": len(overlap),
                    "Status": status,
                })
                print(f"  Overlap({t1}/{split_a} ∩ {t2}/{split_b}): {len(overlap):,} [{status}]")

            overlap_tt = task_keys[t1]["train"] & task_keys[t2]["train"]
            in_remove_scope = family in REMOVE_ON_CONFLICT
            if in_remove_scope:
                status_tt = "DEDUPED" if len(overlap_tt) == 0 else f"{len(overlap_tt)} shared"
            else:
                status_tt = "ALLOWED (no within dedup)" if len(overlap_tt) > 0 else "ALLOWED"
            overlap_rows.append({
                "Family": family,
                "Task A": f"{t1}/train",
                "Task B": f"{t2}/train",
                "Overlap": len(overlap_tt),
                "Status": status_tt,
            })
            print(f"  Overlap({t1}/train ∩ {t2}/train): {len(overlap_tt):,}")
    print()

if overlap_rows:
    df_overlap = pd.DataFrame(overlap_rows)
    print("\n=== Overlap Summary Table ===\n")
    display(df_overlap.style.applymap(
        lambda v: "background-color: #ffcccc" if v == "LEAKAGE!" else
                  "background-color: #ccffcc" if v in ["CLEAN", "DEDUPED", "ALLOWED"] else
                  "background-color: #fff3cc" if isinstance(v, str) and "ALLOWED" in v else
                  "background-color: #fff3cc" if isinstance(v, str) and "shared" in v else "",
        subset=["Status"]
    ))


## 6. 전체 요약

### Pipeline 단계별 샘플 수 변화

```
Source Datasets (Raw HF)
  │
  ▼  Step 1: Download & Process  →  Raw/{tag}/step1/{task}_subtask-{i}_{split}/
  │  - SMILES 파싱, Canonical 변환, Graph 변환
  │  - 파싱 실패 시 해당 sample 제거 (return None)
  │  - Arrow 직렬화 실패 시 continue
  │
  ▼  Step 2: Cross-Source Decontamination  →  Raw/{tag}/step2/{task}_subtask-{i}_{split}/
  │  2a. Eval Blacklist: test(+val) entity key 수집
  │  2b. Eval Leakage Removal: train에서 blacklist entity 제거
  │  2c. Within-Family Dedup: reaction family 2개만 (anchor vs remove)
  │  완료 시 `.step2_done` 마커 파일 생성
  │
  ▼  Step 3: Concatenate & Map  →  Processed/{tag}/{Train,Val,Test}/
  │  - per-task concat → .map(prepare_data_instance)
  │  - Prompt formatting (LLaDA template: <|start_header_id|>system..., etc.)
  │  - `<SMILES>...</SMILES>` 유지 + `<SELFIES>...</SELFIES>` 듀얼 컬럼 생성
  │  - SELFIES 변환 실패 sample (`_selfies_valid=False`) .filter로 제거
  │
  ▼  Final Dataset (Train / Val / Test) — dual SMILES+SELFIES columns in single dir
```


In [ ]:
# ---------------------------------------------------------------------------
# 전체 요약 테이블
# ---------------------------------------------------------------------------
print("=" * 70)
print(f"MolDA Dataset Generation — 전체 요약  [{DATA_TAG}]")
print("=" * 70)

print(f"\n[Tasks]")
print(f"  총 Task 수: {len(task_counts)}")
print(f"  Source 수: {len(set(TASK_TO_SOURCE.values()))}")
print(f"  Category 수: {len(set(TASK_CATEGORY.values()))}")

print(f"\n[Per-Task Totals — {primary_label}]")
print(f"  Train: {df_tasks['Train'].sum():>12,}")
print(f"  Val:   {df_tasks['Val'].sum():>12,}")
print(f"  Test:  {df_tasks['Test'].sum():>12,}")
print(f"  Total: {df_tasks['Total'].sum():>12,}")

print(f"\n[Final Concatenated Dataset — {PROCESSED_ROOT}]")
for _, row in df_final.iterrows():
    print(f"  {row['Split']:<5}: {int(row['Samples']):>12,}")
print(f"  Total: {df_final['Samples'].sum():>12,}")

print(f"\n[Entity Families]")
print(f"  Family 수: {len(family_tasks)}")
print(f"  Family 소속 task: {sum(len(v) for v in family_tasks.values())}개")
print(f"  독립 task: {len(independent_tasks)}개")
print(f"  Within-family dedup 대상: {len(REMOVE_ON_CONFLICT)} families "
      f"({list(REMOVE_ON_CONFLICT.keys())})")

# Rejection 요약 (train만)
df_rej_train_ok = df_rej_valid[df_rej_valid["Split"] == "train"]
if len(df_rej_train_ok) > 0:
    total_raw = df_rej_train_ok["Raw (Source)"].sum()
    total_after = df_rej_train_ok["After Pipeline"].sum()
    total_rejected = df_rej_train_ok["Rejected"].sum()
    overall_rate = total_rejected / total_raw * 100 if total_raw > 0 else 0
    print(f"\n[Preprocessing Rejection (Train, Raw → {primary_label})]")
    print(f"  Raw (Source): {int(total_raw):>12,}")
    print(f"  After Pipeline: {int(total_after):>12,}")
    print(f"  Rejected: {int(total_rejected):>12,}")
    print(f"  Overall Rejection Rate: {overall_rate:.2f}%")

print(f"\n{'=' * 70}")


## 7. 최종 데이터셋 구성 — Task별 Rejection 원인 해설

아래 테이블은 각 Task의 원본(Raw Source) → 최종(After Pipeline) 변환 과정에서
**왜 얼마나 탈락했는지**를 구체적으로 설명합니다.

### Rejection 원인 분류

- **DESIGN_SKIP**: 의도적으로 처리하지 않음 (예: name_conversion test/val)
- **VAL_SPLIT**: train에서 validation 분리 (Mol-Instructions의 `train_test_split(test_size=0.02)`) — 손실 아님
- **EVAL_LEAKAGE**: Step 2b에서 eval entity가 train에 있어 제거
- **FAMILY_DEDUP**: Step 2c에서 REMOVE_ON_CONFLICT에 지정된 task가 anchor와 중복되어 제거
- **SMILES_FAIL**: Step 1에서 SMILES 파싱 / SELFIES 디코딩 / Graph 변환 실패


In [ ]:
# ---------------------------------------------------------------------------
# 최종 데이터셋 구성: Task별 Rejection 원인 해설 테이블
# ---------------------------------------------------------------------------

# Family 역매핑: family_name → anchor task 목록
FAMILY_ANCHOR = {}
for family in set(ENTITY_FAMILIES.values()):
    members = [t for t, f in ENTITY_FAMILIES.items() if f == family]
    remove_task = REMOVE_ON_CONFLICT.get(family)
    anchors = [t for t in members if t != remove_task]
    FAMILY_ANCHOR[family] = anchors

MOL_INST_TASKS = {
    "forward_reaction_prediction", "retrosynthesis", "reagent_prediction",
    "qm9_homo", "qm9_lumo", "qm9_homo_lumo_gap",
}
NAME_CONV_TASKS = {"smol-name_conversion-i2s", "smol-name_conversion-s2i"}


def classify_rejection(task, split, raw_count, after_count):
    rejected = raw_count - after_count
    if rejected <= 0:
        return "NONE", "탈락 없음"

    reasons = []

    if task in NAME_CONV_TASKS and split in ("test", "val"):
        return "DESIGN_SKIP", "run.py에서 name_conversion은 train만 처리 (test/val skip)"

    if task in MOL_INST_TASKS and split == "train":
        val_portion = int(raw_count * 0.02)
        if abs(rejected - val_portion) < 50:
            return "VAL_SPLIT", f"train_test_split(test_size=0.02)로 val {val_portion:,}개 분리 (데이터 손실 아님)"

    family = ENTITY_FAMILIES.get(task)
    if family and REMOVE_ON_CONFLICT.get(family) == task and split == "train":
        anchors = FAMILY_ANCHOR.get(family, [])
        reasons.append(f"FAMILY_DEDUP: {family} anchor({', '.join(anchors)})와 entity 중복 → 제거")

    if split == "train" and family:
        reasons.append("EVAL_LEAKAGE: test/val entity key가 train에 존재 → 제거")

    if split == "train":
        reasons.append("SMILES_FAIL: (일부) SMILES 파싱/Graph 변환 실패 가능")

    if not reasons:
        reasons.append("UNKNOWN")

    primary = reasons[0].split(":")[0]
    explanation = " + ".join(reasons)
    return primary, explanation


explain_rows = []
for task in sorted(task_counts.keys()):
    processed = task_counts[task]
    raw = raw_source_counts.get(task, {})

    for split in ["train", "val", "test"]:
        after_count = processed.get(split, 0)
        raw_split = split if split != "val" else "validation"
        raw_count = raw.get(raw_split)
        if raw_count is None and split == "val" and task in MOL_INST_TASKS:
            raw_train = raw.get("train", 0)
            raw_count = int(raw_train * 0.02) if raw_train > 0 else None
        if raw_count is None:
            continue

        rejected = raw_count - after_count
        if rejected == 0 and split != "train":
            continue

        reason, explanation = classify_rejection(task, split, raw_count, after_count)
        rejection_pct = (rejected / raw_count * 100) if raw_count > 0 else 0

        explain_rows.append({
            "Task": task,
            "Source": TASK_TO_SOURCE.get(task, "?"),
            "Split": split,
            "Raw": raw_count,
            "After Pipeline": after_count,
            "Rejected": rejected,
            "Rate": rejection_pct,
            "Primary Cause": reason,
            "Explanation": explanation,
        })

df_explain = pd.DataFrame(explain_rows)
df_explain_train = df_explain[df_explain["Split"] == "train"].sort_values(
    "Rejected", ascending=False
).reset_index(drop=True)

print("=== 최종 데이터셋 구성: Train Split — Rejection 원인 ===\n")
display(df_explain_train.style.format({
    "Raw": "{:,}", "After Pipeline": "{:,}", "Rejected": "{:,}", "Rate": "{:.2f}%",
}).set_properties(**{"text-align": "left"}, subset=["Primary Cause", "Explanation"]))


In [ ]:
# ---------------------------------------------------------------------------
# 특이 케이스: Test/Val split
# ---------------------------------------------------------------------------
df_explain_special = df_explain[
    (df_explain["Split"] != "train") & (df_explain["Rejected"] != 0)
].sort_values("Rejected", ascending=False).reset_index(drop=True)

if len(df_explain_special) > 0:
    print("=== 특이 케이스: Test/Val Split의 Rejection ===\n")
    display(df_explain_special.style.format({
        "Raw": "{:,}", "After Pipeline": "{:,}", "Rejected": "{:,}", "Rate": "{:.2f}%",
    }).set_properties(**{"text-align": "left"}, subset=["Primary Cause", "Explanation"]))
else:
    print("Test/Val split에서 추가 rejection 없음.")


In [ ]:
# ---------------------------------------------------------------------------
# 최종 데이터셋 구성 요약: 각 Task가 어떻게 최종 데이터셋에 기여하는지
# ---------------------------------------------------------------------------

print("=" * 80)
print(f"최종 데이터셋 구성 요약  [{DATA_TAG}]")
print("=" * 80)

summary_rows = []
for source_label, tasks in SOURCE_TASK_MAP.items():
    source_name = source_label.split("\n")[0]
    for task in tasks:
        counts = task_counts.get(task, {"train": 0, "val": 0, "test": 0})
        total = counts["train"] + counts["val"] + counts["test"]

        raw = raw_source_counts.get(task, {})
        raw_train = raw.get("train", None)

        if raw_train is not None and raw_train > counts["train"]:
            _, explanation = classify_rejection(task, "train", raw_train, counts["train"])
            reason_short = explanation.split(":")[0]
        else:
            reason_short = "-"

        family = ENTITY_FAMILIES.get(task, "-")
        if family != "-":
            role = "REMOVE" if REMOVE_ON_CONFLICT.get(family) == task else "ANCHOR"
        else:
            role = "-"

        summary_rows.append({
            "Source": source_name,
            "Task": task,
            "Category": TASK_CATEGORY.get(task, "?"),
            "Train": counts["train"],
            "Val": counts["val"],
            "Test": counts["test"],
            "Total": total,
            "Family": family if family != "-" else "",
            "Role": role if role != "-" else "",
            "Rejection Cause": reason_short,
        })

df_task_summary = pd.DataFrame(summary_rows)

print("\n각 Task의 Source, Family 소속, Dedup Role, 최종 샘플 수:\n")
print("Role: ANCHOR = Family에 속하지만 dedup 시 보존 (또는 MOL2TEXT/TEXT2MOL처럼 둘 다 보존)")
print("      REMOVE = REMOVE_ON_CONFLICT에 지정된 task — anchor와 충돌 시 제거")
print("      (빈칸) = Entity Family 미소속 (독립 task)\n")

display(df_task_summary.style.format({
    "Train": "{:,}", "Val": "{:,}", "Test": "{:,}", "Total": "{:,}",
}).set_properties(**{"text-align": "right"}, subset=["Train", "Val", "Test", "Total"])
 .set_properties(**{"text-align": "center"}, subset=["Role"])
 .applymap(lambda v: "background-color: #e6f3ff" if v == "ANCHOR" else
                      "background-color: #fff3e6" if v == "REMOVE" else "",
           subset=["Role"]))

print(f"\n{'─' * 80}")
print(f"Total Train: {df_task_summary['Train'].sum():>12,}")
print(f"Total Val:   {df_task_summary['Val'].sum():>12,}")
print(f"Total Test:  {df_task_summary['Test'].sum():>12,}")
print(f"Total:       {df_task_summary['Total'].sum():>12,}")
print(f"{'─' * 80}")

print("\n[Source별 기여도 (Train 기준)]")
total_train_all = df_task_summary["Train"].sum()
for source in df_task_summary["Source"].unique():
    mask = df_task_summary["Source"] == source
    train_sum = df_task_summary.loc[mask, "Train"].sum()
    pct = train_sum / total_train_all * 100 if total_train_all > 0 else 0
    n_tasks = mask.sum()
    print(f"  {source:25s}: {train_sum:>12,} ({pct:5.1f}%) — {n_tasks} tasks")
